In [1]:
%pylab inline
import gc
import os
import glob
import numpy as np

import matplotlib
import matplotlib.pyplot as plt

import astropy.io.fits as pyfits
import astropy.table as astTable

import catutil as utilities

Populating the interactive namespace from numpy and matplotlib


In [14]:
version='v2'
inDir='wrkDir/database/s19-Hironao-2/fields/'
nomDir='wrkDir/s19/s19-1/anaCat_newS19Mask_fdeltacut/catalog_obs_reGaus_no_m/'
calibDir='wrkDir/s19/s19-1/anaCat_newS19Mask_fdeltacut/catalog_obs_reGaus_calibrated/'
pzDir='wrkDir/s19/s19-1/anaCat_newS19Mask_fdeltacut/catalog_others/'
pzDir0='wrkDir/database/s19-photoz/sql/'
pozDir='wrkDir/database/s19-poz/origin/'
tractDir='wrkDir/s19/s19-1/anaCat_newS19Mask_fdeltacut/catalog_tracts/'
fieldname='AEGIS'
nameDict={'dempz':'PDz.target_wide_s19a_%s.fits',
          'dnnz':'dNNz_target_wide_s19a_%s.fits',
          'mizuki':'target_wide_s19a_%s_all.fits'
         }

# nom  and calibrated

In [19]:
# field-mask1
data=astTable.Table.read(os.path.join(inDir,'%s.fits' %fieldname))
zdat=astTable.Table.read(os.path.join(inDir,'%s_pz.fits' %fieldname))
data['dnnz_photoz_best']      =   zdat['dnnz_photoz_best']
data['dnnz_photoz_conf_best'] =   zdat['dnnz_photoz_conf_best']
data['dnnz_photoz_std_best']  =   zdat['dnnz_photoz_std_best']
data['dnnz_photoz_err95_min'] =   zdat['dnnz_photoz_err95_min']
data['dnnz_photoz_err95_max'] =   zdat['dnnz_photoz_err95_max']
wlflag=utilities.update_wl_flags(data)['weak_lensing_flag']
wlflag&=  utilities.get_FDFC_flag(data,'wrkDir/database/s19a_fdfc_hp_contarea_izy-gt-5_trimmed_fd001.fits')

data=data[wlflag]
data    =   utilities.update_reGaus_calibration(data)
mask    =   (~np.isnan(data['i_hsmshaperegauss_derived_shear_bias_c1']))
mask    &=  (~np.isnan(data['i_hsmshaperegauss_derived_shear_bias_c2']))
mask    &=  (~np.isnan(data['i_hsmshaperegauss_derived_shear_bias_m']))

data    =   data[mask]
data.write(os.path.join(calibDir,'%s_calibrated.fits' %fieldname),overwrite=True)
data.remove_column('i_hsmshaperegauss_derived_shear_bias_m')
data.write(os.path.join(nomDir,'%s_no_m.fits' %fieldname))

In [13]:
#tract
outDir2=os.path.join(tractDir,'%s_tracts' %fieldname)
if not os.path.isdir(outDir2):
    os.mkdir(outDir2)
data=pyfits.getdata(os.path.join(nomDir,'%s_no_m.fits' %fieldname))
inds=np.arange(len(data))
tractList=list(set(data['tract']))
for tract in tractList:
    indsO=inds[data['tract']==tract]
    dataO=data[indsO]
    pyfits.writeto(os.path.join(outDir2,'%s_no_m.fits'%tract),dataO)
    del dataO,indsO
    gc.collect()

# Potho-z(new)

In [10]:
#field
data=astTable.Table.read(os.path.join(nomDir,'%s_no_m.fits' %fieldname))
zdat2=astTable.Table.read(os.path.join(pzDir0,'%s.fits' %fieldname))
ids=data['object_id',]
join=astTable.join(left=zdat2,right=ids,keys='object_id',join_type='right')
join.write(os.path.join(pzDir,'%s_pz_2.fits' %fieldname),overwrite=True)

In [19]:
#tract
outDir2=os.path.join(tractDir,'%s_tracts' %fieldname)
if not os.path.isdir(outDir2):
    os.mkdir(outDir2)
pata=pyfits.getdata(os.path.join(pzDir,'%s_pz_2.fits' %fieldname))
data=pyfits.getdata(os.path.join(nomDir,'%s_no_m.fits'%fieldname))
assert np.all(pata['object_id']==data['object_id'])
inds=np.arange(len(data))
tractList=list(set(data['tract']))
for tract in tractList:
    indsO=inds[data['tract']==tract]
    pataO=pata[indsO]
    pyfits.writeto(os.path.join(outDir2,'%s_pz.fits'%tract),pataO)
    del pataO,indsO
    gc.collect()

# poz

In [18]:
tractList=[16821,16822,16972,16973]
for tract in tractList:
    fname=os.path.join(tractDir,'%s_tracts' %fieldname,'%s_no_m.fits' %(tract))
    assert os.path.isfile(fname)
    ids=astTable.Table.read(fname)['object_id',]
    ids.rename_column('object_id','ID')
    for key in nameDict.keys():
        oname=os.path.join(tractDir,'%s_tracts' %fieldname,'%s_pz_pdf_%s.fits' %(tract,key))
        pzname=os.path.join(pozDir,key,nameDict[key] %(tract))
        assert os.path.isfile(pzname)
        fits=astTable.Table.read(pzname,hdu=1)
        join=astTable.join(left=fits,right=ids,keys='ID',join_type='right')
        assert np.all(join['ID']==ids['ID'])
        join.write(oname,overwrite=True)
        del join,fits,pzname,oname
        gc.collect()
    del ids,fname
    gc.collect()

In [ ]:
# Masked to nan
tractList=[16821,16822,16972,16973]
for tract in tractList:
    for key in nameDict.keys():
        oname=os.path.join(tractDir,'%s_tracts' %fieldname,'%s_pz_pdf_%s.fits' %(tract,key))
        assert os.path.isfile(oname)
        fits=pyfits.getdata(oname)
        mask=fits['PDF'][:,0]<1e10
        print(np.sum(mask)/len(mask))
        ids=np.arange(len(mask))[~mask]
        for i in ids:
            fits[i]['PDF']=np.nan
        pyfits.writeto(oname,fits,overwrite=True)
        del mask,fits,ids,oname
        gc.collect()

In [ ]:
# Second HDU
tractList=[16821,16822,16972,16973]
for tract in tractList:
    for key in nameDict.keys():
        pzname=os.path.join(pozDir,key,nameDict[key] %(tract))
        assert os.path.isfile(pzname)
        patade=pyfits.open(pzname)[2]
        
        oname=os.path.join(tractDir,'%s_tracts' %fieldname,'%s_pz_pdf_%s.fits' %(tract,key))
        assert os.path.isfile(oname)
        fits=pyfits.open(oname)
        fits.append(patade)
        fits.writeto(oname,overwrite=True)
        del fits,oname,patade,pzname
        gc.collect()